# Staged relations in A and D_r on the zero-branch ideal images

This notebook tests the proposed modulo-81 zero-branch relations using the completed source archives at modulus $3^7=2187$. 

Every division witness, including the final $\rho$, lies in $I=(9,T_2)M_d^{(-1)^q}$. The ordinary operator is $T=T_2$ for $q=0$ and $T=-T_2$ for $q=1$.

On the target $J=9P+T_2P$, the intended operators are $A=T_2/9$, $C=A^3-A$, and $D_r=(C^2+3h_r(A)C)/9$. Here these are interpreted by simultaneous staged equations, not by arbitrary matrix division. Propagation requires the separate image-source recursive-transfer argument and its compatibility hypotheses; finite success alone does not establish it.


In [1]:
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from hecke_congruences import *
from load_source_data import load_ideal_source_data
from verify_hecke_relations import relation_spec, verify_hecke_relations_nim

USE_ARCHIVED_SOURCE_DATA = True
MAX_HOWELL_DIMENSION = 4096
SOURCE_DATA_DIRECTORY = Path("source_data")
IDEAL_SOURCE_ARCHIVE = SOURCE_DATA_DIRECTORY / "p3_ideal_9_T2_mod2187"

def get_source_data(R, d, q):
    if USE_ARCHIVED_SOURCE_DATA:
        return load_ideal_source_data(
            R, d, q, IDEAL_SOURCE_ARCHIVE / f"degree_{d}.npz"
        )
    return prepare_source_data(
        R, d, q,
        hecke_indices=(2,),
        ideal={"scalar": 9, "generators": [[[1, [1]]]]},
        recursive=True,
    )


## Coefficient ring, degree range, and exact polynomials

The coefficients of $h_r$ below are the displayed lifts modulo **9**, not merely modulo 3. The selector exponent is 6 and each allowed-root factor has exponent 2; there is no additional square.


In [2]:
p = 3
m = 7
modulus = p^m
R = Integers(modulus)

S.<T,A,D> = PolynomialRing(R)

period = euler_phi(p^m)
relation_period = 54
a_m = p^m * (p - 1)
b_m = p^(m - 1) * (p + 1)
surjectivity_bound = a_m + b_m

base_degrees = tuple(
    d for d in range(0, a_m + b_m, 2) if d%6 == 2
)
degree_residues = tuple(sorted({d%relation_period for d in base_degrees}))
exact_induction_base = tuple(
    d for d in range(b_m, a_m + b_m, 2) if d%6 == 2
)

h = {}
allowed_roots = {}
for r in (2,20,38):
    h[r] = 1 - A^2
    allowed_roots[r] = ((0,2), (0,1), (0,1))
for r in (8,26,44):
    h[r] = 1 - A^2
    allowed_roots[r] = ((0,2), (1,2), (1,2))
h[14] = 2 + 5*A
allowed_roots[14] = ((0,2), (0,2), (0,1))
for r in (32,50):
    h[r] = 2 + A^2
    allowed_roots[r] = ((0,2), (0,1), (0,1))

C = A^3 - A
Q_D = {r: C^2 + 3*h[r]*C for r in degree_residues}
F_D = (D^3 - D)^2
F_branch = {
    (r,s): (1-(A-s)^2)^6 * prod((D-delta)^2 for delta in allowed_roots[r][s])
    for r in degree_residues for s in range(0,3)
}

assert F_D == D^6 - 2*D^4 + D^2
assert F_branch[14,0] == (1-A^2)^6 * (D*(D-2))^2
assert F_branch[14,1] == (1-(A-1)^2)^6 * (D*(D-2))^2
assert F_branch[14,2] == (1-(A-2)^2)^6 * (D*(D-1))^2

print("Dickson degrees:", a_m, b_m)
print("degree residues:", degree_residues)
print("full finite verification range:", base_degrees)
print("exact induction base:", exact_induction_base)


Dickson degrees: 4374 2916
degree residues: (2, 8, 14, 20, 26, 32, 38, 44, 50)
full finite verification range: (2, 8, 14, 20, 26, 32, 38, 44, 50, 56, 62, 68, 74, 80, 86, 92, 98, 104, 110, 116, 122, 128, 134, 140, 146, 152, 158, 164, 170, 176, 182, 188, 194, 200, 206, 212, 218, 224, 230, 236, 242, 248, 254, 260, 266, 272, 278, 284, 290, 296, 302, 308, 314, 320, 326, 332, 338, 344, 350, 356, 362, 368, 374, 380, 386, 392, 398, 404, 410, 416, 422, 428, 434, 440, 446, 452, 458, 464, 470, 476, 482, 488, 494, 500, 506, 512, 518, 524, 530, 536, 542, 548, 554, 560, 566, 572, 578, 584, 590, 596, 602, 608, 614, 620, 626, 632, 638, 644, 650, 656, 662, 668, 674, 680, 686, 692, 698, 704, 710, 716, 722, 728, 734, 740, 746, 752, 758, 764, 770, 776, 782, 788, 794, 800, 806, 812, 818, 824, 830, 836, 842, 848, 854, 860, 866, 872, 878, 884, 890, 896, 902, 908, 914, 920, 926, 932, 938, 944, 950, 956, 962, 968, 974, 980, 986, 992, 998, 1004, 1010, 1016, 1022, 1028, 1034, 1040, 1046, 1052, 1058, 1064, 1070, 

## Verify the four staged identities with Nim

For each cyclic generator, the verifier constructs equations inside $I$: every application of $A$ introduces $Ty=9z$; every application of $D_r$ introduces $Q_D(A)y=9z$. The terminal vector must equal $3\rho$ with $\rho\in I$.

The four polynomials are submitted together, sharing the loaded source and its ordinary Hecke matrix. The bridge applies the orientation twist exactly once; the notebook does not pre-twist the stored matrix.

Expanded monomials $A^iD^j$ are evaluated with $D$ first, then $A$. The default independent-monomial semantics follows the manuscript: different monomials may use different intermediate witnesses. The fast route first tries and replays shared-chain witnesses. If those choices fail, the Howell fallback solves the independent-monomial system, including the nested numerator. Each terminal identity has its own witness system.

A Howell success proves existence without exporting witnesses. `MAX_HOWELL_DIMENSION` guards its dense allocation; exceeding it reports `inconclusive` and stops the loop, not a mathematical obstruction. Other process/resource errors also stop execution. All-weight image-source propagation remains a separate argument.

Build the native verifier once from the repository root with `./build_verify_hecke_relations.sh`. The notebook uses that executable without compiling or regenerating archived sources.

The verification loop uses four workers in fresh process pools, as in the mod-49 selector check. It finishes the active batch and schedules no further cases after a failure or inconclusive result. Both orientations are retained in the lower and induction-base degrees.


In [3]:
def verify_A_D_case(d, q):
    data = get_source_data(R, d, q)
    if not data.get("source_scope", "").startswith("ideal_image"):
        raise ValueError("this test requires the ideal-image source, not the whole Manin quotient")

    r = d % relation_period
    spec = relation_spec(
        [("Frobenius", F_D, 1)] + [
            (f"branch_{s}", F_branch[r,s], 1) for s in range(0,3)
        ],
        hecke_operators={"T": 2},
        divisions={"A": (T, 2), "D": (Q_D[r], 2)},
        witness_semantics="independent_monomials",
        max_howell_dimension=MAX_HOWELL_DIMENSION,
    )
    report = verify_hecke_relations_nim(spec, data)
    tests = {test["name"]: test for test in report["relations"]}

    return {
        "degree": d,
        "residue": r,
        "orientation": q,
        "sign": (-1)^q,
        "rank": report["rank"],
        "coordinate_moduli": tuple(report["coordinate_moduli"]),
        "source_scope": data["source_scope"],
        "working_modulus": modulus,
        "tests": tests,
        "state": report["state"],
        "passed": report["passed"],
        "all_weight_propagation_proved": False,
    }


In [ ]:
from concurrent.futures import (
    ProcessPoolExecutor,
    as_completed,
)

from multiprocessing import get_context

workers = 4

def verify_A_D_worker(case):
    """
    Verify all four staged A,D_r identities in one degree and orientation.

    The worker reuses the loaded source and Hecke matrix for the Nim batch.
    Each batch uses a fresh process pool to release worker memory.
    """
    d, q = case
    return verify_A_D_case(d, q)

lower_base_degrees = tuple(
    d for d in base_degrees if d not in exact_induction_base
)
ordered_degrees = (
    tuple(sorted(exact_induction_base, reverse=True))
    + tuple(sorted(lower_base_degrees, reverse=True))
)
cases = [
    (d, q)
    for d in ordered_degrees
    for q in range(0, p-1)
]

context = get_context("fork")

results = []
failures = []

for start in range(0, len(cases), workers):
    batch = cases[start:start + workers]

    print("——————————————————————————————————————————————")
    print("starting batch:", batch, flush=True)

    with ProcessPoolExecutor(
        max_workers=len(batch),
        mp_context=context,
    ) as executor:

        future_to_case = {
            executor.submit(
                verify_A_D_worker,
                case,
            ): case
            for case in batch
        }

        for future in as_completed(future_to_case):
            d, q = future_to_case[future]

            try:
                test = future.result()

            except Exception as error:
                print(f"ERROR: d={d}, q={q}: {error}", flush=True)
                failures.append({
                    "degree": d,
                    "orientation": q,
                    "state": "error",
                    "error": repr(error),
                })
                continue

            results.append(test)

            for name, relation_test in test["tests"].items():
                print(
                    f"d={d:4d}, "
                    f"r={test['residue']:2d}, "
                    f"q={q}, "
                    f"sign={test['sign']:+d}, "
                    f"rank={test['rank']:4d}, "
                    f"relation={name}, "
                    f"route={relation_test['verification_route']}, "
                    f"passed={relation_test['passed']}",
                    flush=True,
                )

            if test["passed"] is not True:
                failures.append(test)

    # Finish the active batch, but do not schedule more cases after a
    # failed identity, an inconclusive result, or a worker/resource error.
    if failures:
        print("stopping after unsuccessful batch")
        break

print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("failures or inconclusive cases:", len(failures))

if failures:
    for failure in failures:
        print(failure)

    if any(failure["state"] == "failed" for failure in failures):
        raise AssertionError("AT LEAST ONE STAGED RELATION FAILED")

    raise RuntimeError(
        "VERIFICATION INCOMPLETE: a case was inconclusive or a worker failed; "
        "this is not a mathematical obstruction"
    )

print("ALL FOUR FINITE IDEAL-IMAGE IDENTITIES VERIFIED")
print("All-weight image-source propagation is a separate argument.")


——————————————————————————————————————————————
starting batch: [(7286, 0), (7286, 1), (7280, 0), (7280, 1)]
d=7286, r=50, q=0, sign=+1, rank= 608, relation=Frobenius, route=explicit_witness_replay, passed=True
d=7286, r=50, q=0, sign=+1, rank= 608, relation=branch_0, route=explicit_witness_replay, passed=True
d=7286, r=50, q=0, sign=+1, rank= 608, relation=branch_1, route=explicit_witness_replay, passed=True
d=7286, r=50, q=0, sign=+1, rank= 608, relation=branch_2, route=explicit_witness_replay, passed=True
d=7286, r=50, q=1, sign=-1, rank= 607, relation=Frobenius, route=explicit_witness_replay, passed=True
d=7286, r=50, q=1, sign=-1, rank= 607, relation=branch_0, route=explicit_witness_replay, passed=True
d=7286, r=50, q=1, sign=-1, rank= 607, relation=branch_1, route=explicit_witness_replay, passed=True
d=7286, r=50, q=1, sign=-1, rank= 607, relation=branch_2, route=explicit_witness_replay, passed=True
d=7280, r=44, q=1, sign=-1, rank= 606, relation=Frobenius, route=explicit_witness_